# 00 Data Collection from the Europeana API

This notebook documents the original data-collection stage for the Europeana India metadata project.

It retrieves a balanced dataset of approximately 1,500 Europeana records using four thematic query groups:

- geographic
- colonial
- cultural
- material

Each group is limited to 375 records. The resulting dataset is saved as:

`data/raw/europeana_india_dataset_1500_raw.csv`

## Important reproducibility note

Europeana is a live service. Search results, metadata, ranking, indexing, and availability may change over time. Therefore, rerunning this notebook may not reproduce the exact same records even when the same queries and limits are used.

The original CSV stored in `data/raw/` should be treated as the fixed source dataset used for the project analysis.

## API key setup

The Europeana API key is **not stored in this notebook**.

Before running the notebook, set an environment variable named:

`EUROPEANA_API_KEY`

### PowerShell example

```powershell
$env:EUROPEANA_API_KEY="YOUR_API_KEY_HERE"
```

Then launch JupyterLab from the project folder in the same PowerShell session.

Do not commit your real API key to GitHub.

In [10]:
# Import required libraries

import os
import time
from pathlib import Path

import pandas as pd
import requests

print("Libraries imported successfully.")

Libraries imported successfully.


In [11]:
# Define project paths safely

current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_dir = current_folder.parent
else:
    project_dir = current_folder

data_dir = project_dir / "data"
raw_dir = data_dir / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

output_file = raw_dir / "europeana_india_dataset_1500_raw.csv"

print("Current folder:", current_folder)
print("Project folder:", project_dir)
print("Raw-data folder:", raw_dir)
print("Output file:", output_file)


Current folder: C:\Users\kevin\Documents\GitHub\Open_Project_2026\notebooks
Project folder: C:\Users\kevin\Documents\GitHub\Open_Project_2026
Raw-data folder: C:\Users\kevin\Documents\GitHub\Open_Project_2026\data\raw
Output file: C:\Users\kevin\Documents\GitHub\Open_Project_2026\data\raw\europeana_india_dataset_1500_raw.csv


In [12]:
# Load the Europeana API key from the environment

API_KEY = os.getenv("EUROPEANA_API_KEY")

if not API_KEY:
    raise ValueError(
        "EUROPEANA_API_KEY is not set. "
        "Set it in PowerShell before launching JupyterLab."
    )

BASE_URL = "https://api.europeana.eu/record/v2/search.json"

print("Europeana API key detected.")

Europeana API key detected.


## Search strategy

The search strategy is organised into four thematic groups. Each group contains several related search terms and contributes up to 375 records.

The collection is balanced by design:

- geographic: 375
- colonial: 375
- cultural: 375
- material: 375

This produces a target total of 1,500 records.

In [13]:
# Define query groups and collection limits

query_groups = {
    "geographic": [
        "India",
        "Indian",
        "Indien",
        "Indische India",
    ],
    "colonial": [
        "British India",
        "colonial India",
        "British Raj",
        "East India Company",
        "colonial collection India",
    ],
    "cultural": [
        "Indian art",
        "Indian museum collection",
        "South Asia collection",
        "ethnography India",
        "India exhibition",
    ],
    "material": [
        "Indian textile",
        "Indian manuscript",
        "Indian photograph",
        "Indian map",
        "Indian print",
    ],
}

group_limits = {
    "geographic": 375,
    "colonial": 375,
    "cultural": 375,
    "material": 375,
}

print("Query groups configured:")
for group, terms in query_groups.items():
    print(f"- {group}: {len(terms)} search terms, limit {group_limits[group]}")

Query groups configured:
- geographic: 4 search terms, limit 375
- colonial: 5 search terms, limit 375
- cultural: 5 search terms, limit 375
- material: 5 search terms, limit 375


In [14]:
# Helper function for safe API requests

def fetch_page(query, page, rows=100, timeout=30):
    params = {
        "wskey": API_KEY,
        "query": query,
        "rows": rows,
        "page": page,
    }

    response = requests.get(
        BASE_URL,
        params=params,
        timeout=timeout,
    )

    response.raise_for_status()
    return response.json()

In [15]:
# Collect records from Europeana

all_items = []

for group, queries in query_groups.items():
    group_count = 0
    print(f"\nCollecting group: {group}")

    for query in queries:
        page = 1

        while group_count < group_limits[group]:
            try:
                data = fetch_page(query=query, page=page)
            except requests.RequestException as error:
                print(
                    f"Request failed for group={group}, "
                    f"query={query!r}, page={page}: {error}"
                )
                break

            items = data.get("items", [])

            if not items:
                print(f"No more records for query: {query!r}")
                break

            for item in items:
                if group_count >= group_limits[group]:
                    break

                all_items.append({
                    "query_group": group,
                    "search_term": query,
                    "title": (
                        item.get("title", [None])[0]
                        if item.get("title")
                        else None
                    ),
                    "description": (
                        item.get("dcDescription", [None])[0]
                        if item.get("dcDescription")
                        else None
                    ),
                    "subject": (
                        item.get("dcSubject", [None])[0]
                        if item.get("dcSubject")
                        else None
                    ),
                    "dataProvider": (
                        item.get("dataProvider", [None])[0]
                        if item.get("dataProvider")
                        else None
                    ),
                    "type": item.get("type"),
                    "country": item.get("country"),
                })

                group_count += 1

            print(
                f"  {query!r}, page {page}: "
                f"group total = {group_count}"
            )

            page += 1
            time.sleep(0.1)

    print(f"Completed {group}: {group_count} records")


  'India', page 1: group total = 100
  'India', page 2: group total = 200
  'India', page 3: group total = 300
  'India', page 4: group total = 375
Completed geographic: 375 records

  'British India', page 1: group total = 100
  'British India', page 2: group total = 200
  'British India', page 3: group total = 300
  'British India', page 4: group total = 375
Completed colonial: 375 records

  'Indian art', page 1: group total = 100
  'Indian art', page 2: group total = 200
  'Indian art', page 3: group total = 300
  'Indian art', page 4: group total = 375
Completed cultural: 375 records

  'Indian textile', page 1: group total = 100
  'Indian textile', page 2: group total = 200
  'Indian textile', page 3: group total = 300
  'Indian textile', page 4: group total = 375
Completed material: 375 records


In [16]:
# Create the collected dataset

df_collected = pd.DataFrame(all_items)

print("Final dataset shape:", df_collected.shape)

print("\nQuery-group distribution:")
print(df_collected["query_group"].value_counts())

df_collected.head(10)

Final dataset shape: (1500, 8)

Query-group distribution:
query_group
geographic    375
colonial      375
cultural      375
material      375
Name: count, dtype: int64


,query_group,search_term,title,description,subject,dataProvider,type,country
0,geographic,India,India,"Barev.\n28 x 36 cm, na listu 38 x 43 cm\nMěřít...",None,Map Collection UK,IMAGE,[Czech Republic]
1,geographic,India,India,None,None,Internet Culturale,SOUND,[Italy]
2,geographic,India,India,None,None,Internet Culturale,SOUND,[Italy]
3,geographic,India,India,"Characters and interpreters: Banjo; Frisell, B...",None,Internet Culturale,SOUND,[Italy]
4,geographic,India,India.,"Litografie, kolor.\n28,5 x 37 cm na listu 31 x...",None,Map Collection UK,IMAGE,[Czech Republic]
5,geographic,India,India,Barev. 60 x 49 cm na listu 72 x 56 cm Měřítko ...,None,Map Collection UK,IMAGE,[Czech Republic]
6,geographic,India,India,"Kolor.\n25,5 x 35,5 cm nebo menší, sestavená 1...",None,Map Collection UK,IMAGE,[Czech Republic]
7,geographic,India,India.,125 - riders with barded horses and escorts of...,None,Cinecittà - Luce,VIDEO,[Italy]
8,geographic,India,India,None,None,Cineteca di Bologna,TEXT,[Italy]
9,geographic,India,India,"Barev.\n28 x 36 cm, na listu 38 x 43 cm\nMěřít...",None,Map Collection UK,TEXT,[Czech Republic]


In [17]:
# Inspect missing values and object types

print("Missing-value proportions:")
print(df_collected.isna().mean().sort_values(ascending=False))

print("\nObject-type counts:")
print(df_collected["type"].value_counts(dropna=False))

Missing-value proportions:
subject         1.000
description     0.414
search_term     0.000
query_group     0.000
title           0.000
dataProvider    0.000
type            0.000
country         0.000
dtype: float64

Object-type counts:
type
IMAGE    1102
TEXT      332
VIDEO      36
SOUND      30
Name: count, dtype: int64


## Save the collected dataset

The dataset is saved directly as the fixed raw file in the `data/raw/` folder. The next notebook loads this raw dataset and creates the deduplicated analytical dataset.

The existing file is overwritten only when this save cell is deliberately executed.

In [18]:
# Save the collected dataset

df_collected.to_csv(
    output_file,
    index=False,
    encoding="utf-8",
)

print("Dataset saved successfully.")
print("Saved file:", output_file)
print("File exists:", output_file.exists())

Dataset saved successfully.
Saved file: C:\Users\kevin\Documents\GitHub\Open_Project_2026\data\raw\europeana_india_dataset_1500_raw.csv
File exists: True


## Collection-stage summary

This notebook:

1. loads the Europeana API key from an environment variable;
2. defines the four thematic query groups;
3. retrieves up to 375 records per group;
4. creates a structured pandas DataFrame;
5. checks the group distribution and missing values;
6. saves the 1,500-record dataset for the next workflow stage.

The next notebook is:

`notebooks/01_data_access.ipynb`